In [2]:
# # Hello World Cell...

# from pyspark.sql import SparkSession
# import pandas as pd

# # Set up Spark session for Databricks Connect
# spark = SparkSession.builder \
#     .appName("Read Bike Trips") \
#     .getOrCreate()

# # Read the table (same as in Databricks)
# df = spark.read.table("mlops_dev.jobquiro.raw_bike_trips")

# # Get only the first 100 rows
# df = df.limit(100)

# # Convert to pandas (same as in Databricks)
# pdf = df.toPandas()

# # Display first 5 rows (same as in Databricks)
# pdf.head()

In [3]:
# Let's get to the Actual Data Cleaning

# I have a data table called "raw_bike_trips" with columns:
# - Genero_Usuario
# - Edad_Usuario
# - Bici
# - Ciclo_Estacion_Retiro
# - Fecha_Retiro
# - Hora_Retiro
# - Ciclo_EstacionArribo
# - Fecha_Arribo
# - Hora_Arribo

# This table is located on databricks, on the "mlops_dev" database, on the "jobqu" schema.
# I want to create two tables, the training and the test set.
# The goal is to do Demand Forecasting, on each station, for each hour.
# We will start with very few features and enrich them as we go along.

# The only data I initially care is:
# - Ciclo_Estacion_Retiro -> StationID
# - Fecha_Retiro -> Date

In [4]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

# Set up Spark session for Databricks Connect
spark = SparkSession.builder \
    .appName("Read Bike Trips") \
    .getOrCreate()

# Read the table (same as in Databricks)
df = spark.read.table("mlops_dev.jobquiro.raw_bike_trips")

# Print the size of the dataframe
#print(df.count())

# Select only the columns we need
df = df.select("Ciclo_Estacion_Retiro", "Fecha_Retiro", "Hora_Retiro")

# Rename the columns
df = df.withColumnRenamed("Ciclo_Estacion_Retiro", "StationID")
df = df.withColumnRenamed("Fecha_Retiro", "Date")
df = df.withColumnRenamed("Hora_Retiro", "Hour")

# Let's do some quick validation, we should only have data between 2025-05-01 and 2025-08-31
df = df.filter(df['Date'].between('2025-06-01', '2025-08-31'))

# There is something wrong with the data, the HourDate is a datetime, but the Date is wrong.
# For example, when Date is 2025-06-01, the Hour is 2025-09-05 12:20:10
# The Hour should only be 12:20:10, the 2025-09-05 should not be there.
# Let's do the following:
# 1. Combine Date and Hour into a single datetime column (e.g. 2025-06-01 12:20:10) - Call this column "PickUpDateTime"
#   - Convert Date and Hour to strings
#   - Concatenate them
#   - Convert to datetime
# 2. Round the datetime to the nearest hour
# 3. Convert the datetime to a date

# Convert Date and Hour to strings
df = df.withColumn("Date", F.col("Date").cast("string"))
df = df.withColumn("Hour", F.col("Hour").cast("string"))

# Concatenate them, but for Hour we only want the time part (e.g. 12:20:10), only the last 8 characters
df = df.withColumn("PickUpDateTime", F.concat(F.col("Date"), F.substring(F.col("Hour"), -9, 9)))

# Convert PickUpDateTime to datetime
df = df.withColumn("PickUpDateTime", F.to_timestamp("PickUpDateTime"))

# Floor the datetime to the nearest hour
df = df.withColumn("PickUpDateTime", F.date_trunc("hour", F.col("PickUpDateTime")))

# Keep only the columns we need
df = df.select("StationID", "PickUpDateTime")

# Bring a random sample of 1,000 rows and convert to pandas
# Option 1: Use sample with fraction (0.1 = 10% of data)
#sample_df = df.sample(0.1)

# Option 2: Use limit to get exactly 1000 rows (more predictable)
#sample_df = df.limit(1000)
pdf = df.toPandas()

pdf

,StationID,PickUpDateTime
0,173,2025-06-30 23:00:00
1,009,2025-06-30 23:00:00
2,576,2025-06-30 23:00:00
3,365,2025-06-30 23:00:00
4,144,2025-06-30 23:00:00
...,...,...
4853579,034,2025-06-30 23:00:00
4853580,021,2025-06-30 23:00:00
4853581,547,2025-06-30 23:00:00
4853582,547,2025-06-30 23:00:00


In [5]:
#pdf['Date'].value_counts()

# Round to Month + value_counts
pd.to_datetime(pdf['PickUpDateTime']).value_counts()

PickUpDateTime
2025-06-04 18:00:00    6580
2025-07-08 18:00:00    6543
2025-07-01 18:00:00    6504
2025-08-19 18:00:00    6470
2025-06-05 18:00:00    6459
                       ... 
2025-06-16 00:00:00     108
2025-06-12 00:00:00     105
2025-06-03 00:00:00      88
2025-06-30 00:00:00      67
2025-08-26 00:00:00      65
Name: count, Length: 1840, dtype: int64

In [6]:
# Count the numbner of rides
agg_rides = pdf.groupby(['StationID', 'PickUpDateTime']).size().reset_index(name='Rides')

agg_rides.sort_values(by='Rides', ascending=False)

,StationID,PickUpDateTime,Rides
403976,271-272,2025-07-09 07:00:00,162
404212,271-272,2025-07-21 07:00:00,153
404385,271-272,2025-07-30 07:00:00,141
404835,271-272,2025-08-22 08:00:00,138
404404,271-272,2025-07-31 07:00:00,136
...,...,...,...
804891,579,2025-08-11 08:00:00,1
558592,393,2025-07-11 06:00:00,1
804895,579,2025-08-11 13:00:00,1
93997,058,2025-06-16 18:00:00,1


In [7]:
from tqdm import tqdm

# Let's add the missing slots of time
location_ids = agg_rides['StationID'].unique()

# Building the complete time series
full_range = pd.date_range(agg_rides['PickUpDateTime'].min(), agg_rides['PickUpDateTime'].max(), freq='h')

output = pd.DataFrame()

for location_id in tqdm(location_ids):
    location_df = agg_rides[agg_rides['StationID'] == location_id]
    location_df = location_df.set_index('PickUpDateTime')
    location_df.index = pd.DatetimeIndex(location_df.index)
    location_df = location_df.reindex(full_range, fill_value=0)

    # Add back StationID
    location_df['StationID'] = location_id

    # Append to the output
    output = pd.concat([output, location_df])

# Move the index to a column, name it as "Date"
agg_rides_all_slots = output.reset_index(names=['Date'])

In [8]:
agg_rides_all_slots

,Date,StationID,Rides
0,2025-06-01 00:00:00,001,2
1,2025-06-01 01:00:00,001,0
2,2025-06-01 02:00:00,001,0
3,2025-06-01 03:00:00,001,0
4,2025-06-01 04:00:00,001,0
...,...,...,...
1497019,2025-08-31 19:00:00,711,0
1497020,2025-08-31 20:00:00,711,5
1497021,2025-08-31 21:00:00,711,0
1497022,2025-08-31 22:00:00,711,0


In [9]:
from typing import Optional, List
import plotly.express as px

def plot_rides(
    rides: pd.DataFrame,
    locations: Optional[List[int]] = None
    ):
    """
    Plot time-series data
    """
    rides_to_plot = rides[rides.StationID.isin(locations)] if locations else rides

    display(rides_to_plot)
    fig = px.line(
        rides_to_plot,
        x="Date",
        y="Rides",
        color='StationID',
        template='none',
    )

    fig.show()

In [10]:
plot_rides(agg_rides_all_slots, locations=['271-272', '001'])

,Date,StationID,Rides
0,2025-06-01 00:00:00,001,2
1,2025-06-01 01:00:00,001,0
2,2025-06-01 02:00:00,001,0
3,2025-06-01 03:00:00,001,0
4,2025-06-01 04:00:00,001,0
...,...,...,...
582907,2025-08-31 19:00:00,271-272,6
582908,2025-08-31 20:00:00,271-272,16
582909,2025-08-31 21:00:00,271-272,11
582910,2025-08-31 22:00:00,271-272,7


In [11]:
agg_rides_all_slots

,Date,StationID,Rides
0,2025-06-01 00:00:00,001,2
1,2025-06-01 01:00:00,001,0
2,2025-06-01 02:00:00,001,0
3,2025-06-01 03:00:00,001,0
4,2025-06-01 04:00:00,001,0
...,...,...,...
1497019,2025-08-31 19:00:00,711,0
1497020,2025-08-31 20:00:00,711,5
1497021,2025-08-31 21:00:00,711,0
1497022,2025-08-31 22:00:00,711,0


In [12]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int,
    step_size: int
    ) -> list:

        stop_position = len(data) - 1
        
        # Start the first sub-sequence at index position 0
        subseq_first_idx = 0
        subseq_mid_idx = n_features
        subseq_last_idx = n_features + 1
        indices = []
        
        while subseq_last_idx <= stop_position:
            indices.append((subseq_first_idx, subseq_mid_idx, subseq_last_idx))
            
            subseq_first_idx += step_size
            subseq_mid_idx += step_size
            subseq_last_idx += step_size

        return indices

In [13]:
agg_rides_all_slots.head(2)

,Date,StationID,Rides
0,2025-06-01 00:00:00,001,2
1,2025-06-01 01:00:00,001,0


In [14]:
import numpy as np

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'Date', 'Rides', 'StationID'}

    location_ids = ts_data['StationID'].unique()
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for location_id in tqdm(location_ids):

        # keep only ts data for this `location_id`
        ts_data_one_location = ts_data.loc[
            ts_data.StationID == location_id, 
            ['Date', 'Rides']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        pickup_hours = []
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['Rides'].values
            y[i] = ts_data_one_location.iloc[idx[1]:idx[2]]['Rides'].values.item()
            # Ensure we preserve the datetime information
            pickup_hour = ts_data_one_location.iloc[idx[1]]['Date']
            # Convert to datetime if it's not already
            if not pd.api.types.is_datetime64_any_dtype(type(pickup_hour)):
                pickup_hour = pd.to_datetime(pickup_hour)
            pickup_hours.append(pickup_hour)

        # numpy -> pandas
        features_one_location = pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        features_one_location['pickup_hour'] = pickup_hours
        features_one_location['pickup_location_id'] = location_id

        # numpy -> pandas
        targets_one_location = pd.DataFrame(y, columns=[f'target_rides_next_hour'])

        # concatenate results
        features = pd.concat([features, features_one_location])
        targets = pd.concat([targets, targets_one_location])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_rides_next_hour']

In [15]:
features, targets = transform_ts_data_into_features_and_target(
    agg_rides_all_slots,
    input_seq_len=24*7*1, # one week of history
    step_size=24,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

features.shape=(57630, 170)
targets.shape=(57630,)


In [16]:
features['pickup_hour']

0       2025-06-08
1       2025-06-09
2       2025-06-10
3       2025-06-11
4       2025-06-12
           ...    
57625   2025-08-27
57626   2025-08-28
57627   2025-08-29
57628   2025-08-30
57629   2025-08-31
Name: pickup_hour, Length: 57630, dtype: datetime64[ns]

In [17]:
# Check the pickup_hour column
print("Sample of pickup_hour values:")
print(features['pickup_hour'].head(10))
print("\nData type of pickup_hour:")
print(features['pickup_hour'].dtype)
print("\nSample of pickup_hour values (last 10):")
print(features['pickup_hour'].tail(10))

Sample of pickup_hour values:
0   2025-06-08
1   2025-06-09
2   2025-06-10
3   2025-06-11
4   2025-06-12
5   2025-06-13
6   2025-06-14
7   2025-06-15
8   2025-06-16
9   2025-06-17
Name: pickup_hour, dtype: datetime64[ns]

Data type of pickup_hour:
datetime64[ns]

Sample of pickup_hour values (last 10):
57620   2025-08-22
57621   2025-08-23
57622   2025-08-24
57623   2025-08-25
57624   2025-08-26
57625   2025-08-27
57626   2025-08-28
57627   2025-08-29
57628   2025-08-30
57629   2025-08-31
Name: pickup_hour, dtype: datetime64[ns]


In [18]:
# Let's check the original Date column in agg_rides_all_slots
print("Sample of Date column in agg_rides_all_slots:")
print(agg_rides_all_slots['Date'].head(10))
print("\nData type of Date column:")
print(agg_rides_all_slots['Date'].dtype)
print("\nSample of Date column (last 10):")
print(agg_rides_all_slots['Date'].tail(10))


Sample of Date column in agg_rides_all_slots:
0   2025-06-01 00:00:00
1   2025-06-01 01:00:00
2   2025-06-01 02:00:00
3   2025-06-01 03:00:00
4   2025-06-01 04:00:00
5   2025-06-01 05:00:00
6   2025-06-01 06:00:00
7   2025-06-01 07:00:00
8   2025-06-01 08:00:00
9   2025-06-01 09:00:00
Name: Date, dtype: datetime64[ns]

Data type of Date column:
datetime64[ns]

Sample of Date column (last 10):
1497014   2025-08-31 14:00:00
1497015   2025-08-31 15:00:00
1497016   2025-08-31 16:00:00
1497017   2025-08-31 17:00:00
1497018   2025-08-31 18:00:00
1497019   2025-08-31 19:00:00
1497020   2025-08-31 20:00:00
1497021   2025-08-31 21:00:00
1497022   2025-08-31 22:00:00
1497023   2025-08-31 23:00:00
Name: Date, dtype: datetime64[ns]


In [29]:
# Regenerate features and targets with the fixed function
features, targets = transform_ts_data_into_features_and_target(
    agg_rides_all_slots,
    input_seq_len=24*7*1, # one week of history
    step_size=6,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

# Check the pickup_hour column again
print("\nSample of pickup_hour values after fix:")
print(features['pickup_hour'].head(10))
print("\nData type of pickup_hour:")
print(features['pickup_hour'].dtype)


features.shape=(230520, 170)
targets.shape=(230520,)

Sample of pickup_hour values after fix:
0   2025-06-08 00:00:00
1   2025-06-08 06:00:00
2   2025-06-08 12:00:00
3   2025-06-08 18:00:00
4   2025-06-09 00:00:00
5   2025-06-09 06:00:00
6   2025-06-09 12:00:00
7   2025-06-09 18:00:00
8   2025-06-10 00:00:00
9   2025-06-10 06:00:00
Name: pickup_hour, dtype: datetime64[ns]

Data type of pickup_hour:
datetime64[ns]


In [30]:
tabular_data = features
tabular_data['target_rides_next_hour'] = targets

In [31]:
tabular_data

,rides_previous_168_hour,rides_previous_167_hour,rides_previous_166_hour,rides_previous_165_hour,rides_previous_164_hour,rides_previous_163_hour,rides_previous_162_hour,rides_previous_161_hour,rides_previous_160_hour,rides_previous_159_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,pickup_location_id,target_rides_next_hour
0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,2.0,13.0,...,16.0,5.0,18.0,8.0,6.0,3.0,1.0,2025-06-08 00:00:00,001,2.0
1,1.0,6.0,2.0,13.0,6.0,10.0,7.0,5.0,12.0,10.0,...,1.0,2.0,0.0,0.0,0.0,0.0,2.0,2025-06-08 06:00:00,001,5.0
2,7.0,5.0,12.0,10.0,15.0,13.0,3.0,6.0,12.0,8.0,...,2.0,5.0,4.0,4.0,18.0,11.0,18.0,2025-06-08 12:00:00,001,14.0
3,3.0,6.0,12.0,8.0,5.0,3.0,0.0,0.0,0.0,0.0,...,18.0,14.0,18.0,17.0,11.0,11.0,11.0,2025-06-08 18:00:00,001,12.0
4,0.0,0.0,0.0,0.0,0.0,3.0,12.0,6.0,17.0,21.0,...,11.0,12.0,9.0,13.0,8.0,6.0,2.0,2025-06-09 00:00:00,001,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230515,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,4.0,1.0,3.0,3.0,0.0,2025-08-30 18:00:00,711,3.0
230516,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,...,0.0,3.0,1.0,1.0,1.0,0.0,0.0,2025-08-31 00:00:00,711,0.0
230517,0.0,0.0,3.0,4.0,2.0,2.0,7.0,5.0,3.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-08-31 06:00:00,711,3.0
230518,7.0,5.0,3.0,8.0,2.0,1.0,1.0,3.0,1.0,2.0,...,0.0,3.0,1.0,3.0,0.0,3.0,2.0,2025-08-31 12:00:00,711,7.0


In [32]:
tabular_data.query('pickup_location_id == "001"')

,rides_previous_168_hour,rides_previous_167_hour,rides_previous_166_hour,rides_previous_165_hour,rides_previous_164_hour,rides_previous_163_hour,rides_previous_162_hour,rides_previous_161_hour,rides_previous_160_hour,rides_previous_159_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,pickup_location_id,target_rides_next_hour
0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,2.0,13.0,...,16.0,5.0,18.0,8.0,6.0,3.0,1.0,2025-06-08 00:00:00,001,2.0
1,1.0,6.0,2.0,13.0,6.0,10.0,7.0,5.0,12.0,10.0,...,1.0,2.0,0.0,0.0,0.0,0.0,2.0,2025-06-08 06:00:00,001,5.0
2,7.0,5.0,12.0,10.0,15.0,13.0,3.0,6.0,12.0,8.0,...,2.0,5.0,4.0,4.0,18.0,11.0,18.0,2025-06-08 12:00:00,001,14.0
3,3.0,6.0,12.0,8.0,5.0,3.0,0.0,0.0,0.0,0.0,...,18.0,14.0,18.0,17.0,11.0,11.0,11.0,2025-06-08 18:00:00,001,12.0
4,0.0,0.0,0.0,0.0,0.0,3.0,12.0,6.0,17.0,21.0,...,11.0,12.0,9.0,13.0,8.0,6.0,2.0,2025-06-09 00:00:00,001,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335,7.0,10.0,4.0,3.0,6.0,5.0,0.0,0.0,0.0,0.0,...,9.0,13.0,9.0,10.0,20.0,8.0,17.0,2025-08-30 18:00:00,001,4.0
336,0.0,0.0,0.0,0.0,0.0,0.0,5.0,7.0,6.0,2.0,...,17.0,4.0,6.0,9.0,3.0,4.0,9.0,2025-08-31 00:00:00,001,0.0
337,5.0,7.0,6.0,2.0,10.0,12.0,17.0,12.0,11.0,14.0,...,9.0,0.0,0.0,0.0,0.0,0.0,2.0,2025-08-31 06:00:00,001,1.0
338,17.0,12.0,11.0,14.0,11.0,12.0,11.0,9.0,9.0,5.0,...,2.0,1.0,4.0,8.0,3.0,7.0,10.0,2025-08-31 12:00:00,001,9.0
